In [92]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("sci.mplstyle")

In [93]:
def ham_doc_file(filename):
    data = np.loadtxt(filename, comments="#")

    t_raw = data[:, 2]
    x_raw = data[:, 3]
    u_raw = data[:, 4]

    x = np.unique(x_raw)
    t = np.unique(t_raw)

    N_x = len(x)
    N_time = len(t)

    print("File:", filename)
    print("N_x =", N_x)
    print("N_time =", N_time)
    print("So diem =", len(u_raw))
    print("N_x * N_time =", N_x * N_time)
    print("\n")

    if N_x * N_time != len(u_raw):
        raise ValueError("File khong phai luoi day du theo x va t")

    # file ghi theo:
    # for j in range(N_time):
    #     for i in range(N_x):
    temp = u_raw.reshape(N_time, N_x)

    return x, t, temp

In [94]:
def ham_ghi_file_calculus(file): # dung de ghi thong tin quan trong ra file

    file.write("# Nghiem giai tich cho phuong trinh truyen nhiet 1D\n")
    file.write("#\n")
    file.write(f"# {'j (t)':>8s} {'i (x)':>8s} {'t':>15s} {'x':>15s} {'u':>15s}\n")

def ham_ghi_file_sosanh(file): # dung de ghi thong tin quan trong ra file

    file.write("# Nghiem giai tich cho phuong trinh truyen nhiet 1D\n")
    file.write("#\n")
    file.write(f"# {'j (t)':>8s} {'i (x)':>8s} {'t':>15s} {'x':>15s} {'u':>15s} {'|err|':>15s}\n")


def ham_luu_dulieu_calculus(filename, u, dk_bai_toan, skip_buoc_thoigian=1):
    N_x = dk_bai_toan["N_x"]
    N_time = dk_bai_toan["N_time"]
    h = dk_bai_toan["h"]
    k = dk_bai_toan["k"]

    with open(filename, "w", encoding="utf-8") as file:
        ham_ghi_file_calculus(file)

        for j in range(N_time):
            if j % skip_buoc_thoigian != 0 and j != N_time - 1: # khi j khong chia het cho skip_buoc_thoigian thi bo qua data do va lay luon  data cuoi cung
                continue
            t = j * k

            for i in range(N_x):
                x = i * h
                file.write(f"  {j:8d} {i:8d} {t:15.6e} {x:15.6e} {u[i, j]:15.6e}\n")

            file.write("\n")

def ham_luu_dulieu_sosanh(filename, u_calculus, u_giaiso, dk_bai_toan, skip_buoc_thoigian=1):
    N_x = dk_bai_toan["N_x"]
    N_time = dk_bai_toan["N_time"]
    h = dk_bai_toan["h"]
    k = dk_bai_toan["k"]

    with open(filename, "w", encoding="utf-8") as file:
        ham_ghi_file_sosanh(file)

        for j in range(N_time):
            if j % skip_buoc_thoigian != 0 and j != N_time - 1: # khi j khong chia het cho skip_buoc_thoigian thi bo qua data do va lay luon  data cuoi cung
                continue
            t = j * k

            for i in range(N_x):
                x = i * h
                err = np.abs(u_calculus[i, j] - u_giaiso[i, j])
                file.write(f"  {j:8d} {i:8d} {t:15.6e} {x:15.6e} {u_calculus[i, j]:15.6e} {err:15.6e}\n")

            file.write("\n")

In [95]:
x_backward_1thanh , t_backward_1thanh, temp_backward_1thanh = ham_doc_file("truyen_nhiet_1thanh_backward_result.txt")

x_forward_1thanh , t_forward_1thanh, temp_forward_1thanh = ham_doc_file("truyen_nhiet_1thanh_forward_result.txt")


File: truyen_nhiet_1thanh_backward_result.txt
N_x = 100
N_time = 301
So diem = 30100
N_x * N_time = 30100


File: truyen_nhiet_1thanh_forward_result.txt
N_x = 100
N_time = 301
So diem = 30100
N_x * N_time = 30100




In [96]:
# Nghiem giai tich cua 1 thanh

def tinh_calculus_1thanh(x_giaiso, time_giaiso, u_giaiso, q_giaitich, filename):
    global alpha, L
    N_x = len(x_giaiso)
    N_time = len(time_giaiso)
    h = x_giaiso[1]-x_giaiso[0]
    k = time_giaiso[1] -time_giaiso[0]
    u = np.zeros((N_x, N_time), dtype=float)
    
    u_giaiso = u_giaiso.T #do khi doc thi da transpose cho ve hinh cho tien, nen transpose lai cho tinh
    # ham doc file giai so define la u_raw.reshape(N_time, N_x)
    # nhung khi luu data thi luu u(N_x, N_time), nen la phai transpose lai

    if np.size(u_giaiso) != np.size(u):
        sosanh = False
    else:
        sosanh = True

    for j in range(N_time):
        for i in range(1, N_x - 1):
            phantu_u = 0.0
            x = x_giaiso[i]
            time = time_giaiso[j]
            for q in range(q_giaitich):
                m = 2 * q + 1  # chi lay mode le
                phantu_u += (
                    400.0 / (m * np.pi)
                    * np.sin(m * np.pi * x / L)
                    * np.exp(-alpha**2 * (m * np.pi / L)**2 * time)
                )
            u[i, j] = phantu_u
    
    dk_bai_toan = {
        "N_x": N_x,
        "N_time": N_time,
        "L": L,
        "h": h,
        "k": k,
    }

    if sosanh == True:
        ham_luu_dulieu_sosanh(filename + "_calculus_result.txt", u, u_giaiso, dk_bai_toan)
    else:
        ham_luu_dulieu_calculus(filename + "_calculus_result.txt", u, dk_bai_toan)

    print("Da tinh xong")
    ten_file = filename + "_calculus_result.txt"
    print(f"Da luu file: {ten_file}")

    print(f"Max err = ", np.max(np.abs(u_giaiso-u)))
    print(f"Mean err = ", np.mean(np.abs(u_giaiso-u)))

    return u

In [97]:
L = 1.0

kappa = 210.0
C = 900.0
rho = 2700.0

alpha = np.sqrt(kappa / (C * rho))
u_calculus = tinh_calculus_1thanh(x_backward_1thanh, t_backward_1thanh, temp_backward_1thanh, q_giaitich=50, filename  = "truyen_nhiet_1thanh")

Da tinh xong
Da luu file: truyen_nhiet_1thanh_calculus_result.txt
Max err =  17.89117426126856
Mean err =  0.011953078154813756
